# **Proyek Data Pipeline Bank Marketing (Databricks Free Edition)**
## **Tahap 0: Akuisisi Data Eksternal, Pembuatan Batch Kronologis & Eksplorasi Data (EDA)**

Notebook ini dijalankan **satu kali di awal** (sebelum menjalankan Lakeflow Job/Workflow `01 -> 02 -> 03`) untuk:
1. Menyiapkan Schema (`workspace.bank_marketing`) dan Unity Catalog Volume (`raw_data`).
2. Mengunduh dataset **Bank Marketing (`bank-full.csv`)** langsung dari tautan HTTPS resmi UCI Machine Learning Repository (dengan fallback ke repositori GitHub apabila koneksi eksternal dibatasi) dan menyimpannya ke Unity Catalog Volume.
3. Memecah `bank-full.csv` secara kronologis menjadi **3 file batch** di folder `incoming/` Volume (`bank_batch_2008.csv`, `bank_batch_2009.csv`, `bank_batch_2010.csv`) untuk simulasi *incremental batch ingestion* di Task Bronze.
4. Melakukan **Exploratory Data Analysis (EDA)** menggunakan **Spark DataFrame** dan **Spark SQL** (skema, statistik deskriptif, pengecekan `NULL` dan nilai `"unknown"`, deteksi duplikasi, serta 4 agregasi eksploratif).

### **Informasi Sumber Data & Pertanyaan Bisnis**

* **Nama Organisasi / Sumber:** UCI Machine Learning Repository (Dataset ID: 222 — *Portuguese Banking Institution*, Moro et al., 2011)
* **Tautan Resmi:** [https://archive.ics.uci.edu/dataset/222/bank+marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)
* **Tautan Unduhan Langsung (HTTPS):** `https://archive.ics.uci.edu/static/public/222/bank+marketing.zip`
* **File yang Digunakan:** `bank-full.csv` (45.211 baris × 17 kolom)
* **Format Data:** CSV dengan pemisah titik koma (`;`) dan header pada baris pertama
* **Periode Data:** Mei 2008 – November 2010 (berurut kronologis)
* **Lisensi:** Creative Commons Attribution 4.0 International (CC BY 4.0)
* **Alasan Pemilihan Kasus:** Dataset memiliki atribut numerik (`age`, `balance`, `duration`, `campaign`), kategorikal (`job`, `marital`, `education`, `contact`), dimensi waktu (`month`, `day`, urutan kronologis 2008–2010), serta target biner `y` (`yes`/`no`) yang memadai untuk ETL Medallion, dashboard analitik, dan pemodelan klasifikasi MLflow.

#### **Pertanyaan Bisnis & Analisis**
1. Bagaimana pengaruh tingkat pendidikan (`education`) dan status pernikahan (`marital`) terhadap tingkat konversi berlangganan deposito (`y`)?
2. Apakah klien dengan saldo tabungan rata-rata tahunan (`balance`) yang lebih tinggi cenderung memiliki tingkat konversi deposito yang lebih tinggi?
3. Bagaimana tren volume kontak dan tingkat konversi kampanye dari waktu ke waktu (Mei 2008 – November 2010) serta hubungan kelompok usia (`age_group`) terhadap keputusan berlangganan?

---
### **1. Konfigurasi Parameter & Persiapan Unity Catalog Volume**

In [ ]:
import csv
import io
import os
import urllib.request
import zipfile
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F

# Widget parameter agar konsisten dengan konfigurasi Databricks Free Edition
dbutils.widgets.text("catalog", "workspace", "Unity Catalog Name")
dbutils.widgets.text("schema", "bank_marketing", "Schema Name")
dbutils.widgets.text("volume_name", "raw_data", "Volume Name")
dbutils.widgets.text(
    "source_url",
    "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip",
    "URL Sumber Eksternal (UCI)",
)
dbutils.widgets.text(
    "fallback_url",
    "https://raw.githubusercontent.com/Haritsst/Bank_Marketing/main/data/bank-full.csv",
    "URL Fallback (GitHub)",
)

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
source_url = dbutils.widgets.get("source_url").strip()
fallback_url = dbutils.widgets.get("fallback_url").strip()

# Buat Schema dan Managed Volume di Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume_name}")

volume_base = f"/Volumes/{catalog}/{schema}/{volume_name}"
raw_dir = f"{volume_base}/raw"
incoming_dir = f"{volume_base}/incoming"
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(incoming_dir, exist_ok=True)

raw_csv_path = f"{raw_dir}/bank-full.csv"
print(f"Schema   : {catalog}.{schema}")
print(f"Volume   : {volume_base}")
print(f"Raw Path : {raw_csv_path}")
print(f"Incoming : {incoming_dir}")

---
### **2. Akuisisi Data `bank-full.csv` dari Sumber Eksternal ke Volume**

In [ ]:
def extract_bank_full_from_zip(zip_bytes):
    """Mengekstrak bank-full.csv dari arsip UCI (mendukung arsip zip bersarang)."""
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for name in zf.namelist():
            if name.lower().endswith("bank-full.csv") and "__macosx" not in name.lower():
                return zf.read(name)
        for name in zf.namelist():
            if name.lower().endswith(".zip") and "__macosx" not in name.lower():
                nested_bytes = zf.read(name)
                try:
                    return extract_bank_full_from_zip(nested_bytes)
                except FileNotFoundError:
                    continue
    raise FileNotFoundError("File bank-full.csv tidak ditemukan di dalam arsip ZIP.")


used_url = None
csv_bytes = None

try:
    print(f"Mengunduh dari sumber resmi UCI: {source_url} ...")
    req = urllib.request.Request(source_url, headers={"User-Agent": "Databricks-ETL-Pipeline/1.0"})
    with urllib.request.urlopen(req, timeout=45) as resp:
        zip_data = resp.read()
    csv_bytes = extract_bank_full_from_zip(zip_data)
    used_url = source_url
    print("Unduhan dari UCI berhasil.")
except Exception as exc:
    print(f"Peringatan: gagal mengunduh dari UCI ({exc}).")
    print(f"Menggunakan URL fallback GitHub: {fallback_url} ...")
    req = urllib.request.Request(fallback_url, headers={"User-Agent": "Databricks-ETL-Pipeline/1.0"})
    with urllib.request.urlopen(req, timeout=45) as resp:
        csv_bytes = resp.read()
    used_url = fallback_url
    print("Unduhan dari URL fallback berhasil.")

with open(raw_csv_path, "wb") as f:
    f.write(csv_bytes)

wib_tz = timezone(timedelta(hours=7))
download_time_wib = datetime.now(wib_tz).strftime("%Y-%m-%d %H:%M:%S WIB")

with open(raw_csv_path, "r", encoding="utf-8-sig") as f:
    line_count = sum(1 for _ in f)
data_row_count = line_count - 1
file_size_bytes = os.path.getsize(raw_csv_path)

print("\n=== METADATA AKUISISI DATA ===")
print(f"Tanggal Pengambilan : {download_time_wib}")
print(f"URL Sumber Dipakai  : {used_url}")
print(f"Lokasi Volume       : {raw_csv_path}")
print(f"Format File         : CSV (delimiter ';')")
print(f"Ukuran File         : {file_size_bytes:,} bytes")
print(f"Jumlah Baris Data   : {data_row_count:,} baris (tidak termasuk header)")

---
### **3. Pemecahan File Batch Kronologis ke Folder `incoming/` (Simulasi Incremental)**

Berdasarkan dokumentasi dataset (`bank-names.txt`), baris pada `bank-full.csv` tersusun secara **kronologis dari Mei 2008 hingga November 2010**.
Sel di bawah ini memecah data asli tanpa mengubah satu pun isi kolom menjadi 3 file batch tahunan (`bank_batch_2008.csv`, `bank_batch_2009.csv`, `bank_batch_2010.csv`) di folder `incoming/` agar dapat di-ingest secara bertahap dan *idempotent* oleh Task Bronze.

In [ ]:
MONTH_TO_NUM = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

# Bersihkan file batch lama di incoming/ saat setup ulang awal
for existing in os.listdir(incoming_dir):
    if existing.startswith("bank_batch_") and existing.endswith(".csv"):
        os.remove(os.path.join(incoming_dir, existing))

batch_handles = {}
batch_counts = {}

try:
    with open(raw_csv_path, "r", encoding="utf-8-sig") as src_f:
        header_line = src_f.readline()
        header_cols = next(csv.reader([header_line], delimiter=";"))
        month_idx = header_cols.index("month")

        current_year = 2008
        prev_month_num = 5  # Dataset dimulai Mei 2008

        for line in src_f:
            if not line.strip():
                continue
            parsed_row = next(csv.reader([line], delimiter=";"))
            month_str = parsed_row[month_idx].strip().lower()
            month_num = MONTH_TO_NUM[month_str]

            if month_num < prev_month_num:
                current_year += 1
            prev_month_num = month_num

            if current_year not in batch_handles:
                batch_path = os.path.join(incoming_dir, f"bank_batch_{current_year}.csv")
                handle = open(batch_path, "w", encoding="utf-8", newline="")
                handle.write(header_line.rstrip("\r\n") + "\n")
                batch_handles[current_year] = handle
                batch_counts[current_year] = 0

            batch_handles[current_year].write(line.rstrip("\r\n") + "\n")
            batch_counts[current_year] += 1
finally:
    for handle in batch_handles.values():
        handle.close()

print("=== FILE BATCH DI FOLDER INCOMING ===")
total_batch_rows = 0
for yr in sorted(batch_counts):
    fname = f"bank_batch_{yr}.csv"
    cnt = batch_counts[yr]
    total_batch_rows += cnt
    print(f"- {fname}: {cnt:,} baris")

print(f"Total seluruh file batch: {total_batch_rows:,} baris")
assert total_batch_rows == data_row_count, "Jumlah baris gabungan batch tidak cocok dengan data mentah!" 

---
### **4. Eksplorasi Data (EDA) Menggunakan Spark DataFrame & Spark SQL**

In [ ]:
# Membaca data mentah dari Unity Catalog Volume ke Spark DataFrame
df_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .csv(raw_csv_path)
)
df_raw.createOrReplaceTempView("bank_raw_eda")

print("=== 4A. Skema Data Mentah ===")
df_raw.printSchema()

total_rows = df_raw.count()
print(f"Total Baris: {total_rows:,} | Total Kolom: {len(df_raw.columns)}")

print("\n=== 4B. Statistik Deskriptif Kolom Numerik & Kategorikal ===")
display(df_raw.describe())

In [ ]:
# Pengecekan SQL NULL, nilai string 'unknown', dan baris duplikat eksak
print("=== 4C. Jumlah SQL NULL per Kolom ===")
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_raw.columns]
display(df_raw.select(null_exprs))

string_cols = [field.name for field in df_raw.schema.fields if field.dataType.simpleString() == "string"]
print("=== 4D. Jumlah Nilai 'unknown' (Missing Terselubung) pada Kolom Kategorikal ===")
unknown_exprs = [
    F.sum((F.lower(F.trim(F.col(c))) == "unknown").cast("int")).alias(c)
    for c in string_cols
]
display(df_raw.select(unknown_exprs))

distinct_rows = df_raw.distinct().count()
exact_duplicates = total_rows - distinct_rows
print("=== 4E. Pengecekan Duplikasi Eksak ===")
print(f"Total baris       : {total_rows:,}")
print(f"Baris unik        : {distinct_rows:,}")
print(f"Baris duplikat    : {exact_duplicates:,}")

In [ ]:
# Minimal 3+ Agregasi Eksploratif dengan Spark SQL (COUNT, AVG, MIN, MAX)
print("=== Agregasi 1: Tingkat Pendidikan & Status Pernikahan vs Konversi Deposito (y) ===")
display(spark.sql("""
    SELECT
        education,
        marital,
        COUNT(*) AS jumlah_klien,
        SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS jumlah_subscribed,
        ROUND(SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS conversion_rate_pct
    FROM bank_raw_eda
    GROUP BY education, marital
    ORDER BY conversion_rate_pct DESC, jumlah_klien DESC
"""))

print("=== Agregasi 2: Statistik Saldo Tahunan (balance) Berdasarkan Keputusan Deposito (y) ===")
display(spark.sql("""
    SELECT
        y,
        COUNT(*) AS jumlah_klien,
        ROUND(AVG(balance), 2) AS avg_balance,
        MIN(balance) AS min_balance,
        MAX(balance) AS max_balance
    FROM bank_raw_eda
    GROUP BY y
    ORDER BY y
"""))

print("=== Agregasi 3: Kelompok Usia vs Hasil Kampanye ===")
display(spark.sql("""
    SELECT
        CASE
            WHEN age < 30 THEN 'Muda (<30)'
            WHEN age BETWEEN 30 AND 50 THEN 'Paruh Baya (30-50)'
            ELSE 'Tua (>50)'
        END AS kelompok_usia,
        COUNT(*) AS jumlah_klien,
        SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS jumlah_subscribed,
        ROUND(SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(AVG(age), 1) AS avg_age
    FROM bank_raw_eda
    GROUP BY kelompok_usia
    ORDER BY conversion_rate_pct DESC
"""))

print("=== Agregasi 4: Ringkasan Aktivitas & Durasi Panggilan per Bulan ===")
display(spark.sql("""
    SELECT
        month,
        COUNT(*) AS total_kontak,
        ROUND(AVG(duration), 2) AS avg_durasi_detik,
        MIN(duration) AS min_durasi_detik,
        MAX(duration) AS max_durasi_detik,
        ROUND(SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS conversion_rate_pct
    FROM bank_raw_eda
    GROUP BY month
    ORDER BY conversion_rate_pct DESC
"""))